# Análise de Série Temporal — Preço do Caroço de Algodão (MT)
### Projeto P2 · ME607 Séries Temporais

Notebook que reproduz toda a análise: carrega os dados, faz a análise exploratória, ajusta o SARIMA, realiza o exercício de previsão e os modelos complementares (híbrido e hierárquico). Cada célula que gera um gráfico salva o `.png` correspondente.

In [ ]:
!pip install -q pymc arch statsmodels 2>/dev/null
print("Dependências instaladas.")

In [ ]:
import pandas as pd, numpy as np
import matplotlib, warnings, os
matplotlib.use('Agg'); warnings.filterwarnings('ignore')
import matplotlib.pyplot as plt, matplotlib.gridspec as gridspec
from matplotlib.ticker import FuncFormatter
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.stats.diagnostic import acorr_ljungbox, het_arch
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from scipy import stats
import pymc as pm, pytensor.tensor as pt
plt.rcParams.update({'figure.facecolor':'white','axes.facecolor':'white','axes.grid':True,
  'grid.alpha':0.25,'axes.spines.top':False,'axes.spines.right':False})
AZUL='#2C6E9B'; VERM='#C0392B'; VERDE='#27AE60'; AMAR='#E67E22'
fmt=FuncFormatter(lambda x,_:f'R${x:,.0f}')
MESES=['Jan','Fev','Mar','Abr','Mai','Jun','Jul','Ago','Set','Out','Nov','Dez']
SEED = 236312
np.random.seed(SEED)
print("Bibliotecas carregadas. Seed =", SEED)

## 1. Carregar o dado único e construir a série mensal (média das safras)

**Origem dos dados:** o preço médio mensal de comercialização do caroço de algodão é
publicado pelo IMEA (Instituto Mato-grossense de Economia Agropecuária). Para obtê-lo,
é preciso acessar o site do IMEA, solicitar o indicador, receber um link por e-mail e
baixar o arquivo. Para facilitar a reprodução deste trabalho, o banco foi espelhado em um
repositório público no GitHub e é lido diretamente da URL abaixo.

In [ ]:
# Lê o banco direto do GitHub (espelho público dos dados do IMEA).
# Caso prefira rodar offline, basta baixar o caroco_p2.csv e usar pd.read_csv('caroco_p2.csv').
URL = 'https://raw.githubusercontent.com/ThalesCrivillari/serie-temporal-caroco-algodao/refs/heads/main/caroco_p2.csv'
try:
    raw = pd.read_csv(URL)
    print('Dados carregados do GitHub.')
except Exception:
    raw = pd.read_csv('caroco_p2.csv')   # fallback: arquivo local
    print('Dados carregados do arquivo local.')

raw['data'] = pd.to_datetime(raw['data'])
raw['am']   = raw['data'].dt.to_period('M')

# Série mensal: média das safras em cada mês
df = (raw.groupby('am')['preco_rs_ton'].mean().reset_index()
      .rename(columns={'preco_rs_ton':'preco'}))
df['data'] = df['am'].dt.to_timestamp()
df['ano']  = df['data'].dt.year; df['mes_num'] = df['data'].dt.month
ts = df['preco'].values; datas = df['data']; n = len(ts); ML = min(15, n//2-2)
print(f"N = {n} meses ({df['am'].iloc[0]} a {df['am'].iloc[-1]})")
print(f"Média R${ts.mean():.0f}  DP R${ts.std():.0f}  Mín R${ts.min():.0f}  Máx R${ts.max():.0f}")

# Diferença média entre safras nos meses de sobreposição (justifica a média)
sob = raw.groupby('am').filter(lambda g: len(g)==2)
difs = [abs(g.sort_values('safra')['preco_rs_ton'].values[0]
            - g.sort_values('safra')['preco_rs_ton'].values[1]) for _,g in sob.groupby('am')]
print(f"Meses com 2 safras: {sob['am'].nunique()} | diferença média entre safras: R${np.mean(difs):.0f}/ton")
df.head()

## 2. Série e decomposição clássica (médias móveis)

In [ ]:
fig,ax=plt.subplots(figsize=(10,3.4))
ax.plot(datas,ts,color=AZUL,lw=1.8); ax.fill_between(datas,ts,alpha=0.1,color=AZUL)
ax.yaxis.set_major_formatter(fmt); ax.set_ylabel('R$/ton')
ax.set_title('Preço médio mensal do caroço de algodão — MT (IMEA)',fontweight='bold')
plt.savefig('f1_serie.png', dpi=150, bbox_inches='tight'); plt.show()

from statsmodels.tsa.seasonal import seasonal_decompose
dec=seasonal_decompose(ts,model='additive',period=12,extrapolate_trend='freq')
stl=dec  # decomposição clássica por médias móveis (Morettin & Toloi)
rr=dec.resid[~np.isnan(dec.resid)]
ss=max(0,1-np.var(rr)/np.var((dec.seasonal+dec.resid)[~np.isnan(dec.resid)]))
tr=max(0,1-np.var(rr)/np.var((dec.trend+dec.resid)[~np.isnan(dec.resid)]))
fig,axes=plt.subplots(3,1,figsize=(10,5.5),sharex=True)
for ax,(lab,c,cor) in zip(axes,[('Tendência',dec.trend,AMAR),('Sazonal',dec.seasonal,VERDE),('Resíduo',dec.resid,VERM)]):
    (ax.bar(datas,c,color=cor,alpha=0.7,width=20) if lab=='Resíduo' else ax.plot(datas,c,color=cor,lw=1.6)); ax.set_ylabel(lab)
axes[0].set_title(f'Decomposição clássica (médias móveis) (força tend.={tr:.2f}, saz.={ss:.2f})',fontweight='bold')
plt.savefig('f2_stl.png', dpi=150, bbox_inches='tight'); plt.show()
print(f"Força tendência={tr:.2f}  sazonalidade={ss:.2f}  (>0.64 = forte)")

## 3. Estacionariedade (ADF + KPSS) e ACF/PACF

In [ ]:
def adf(x,l): r=adfuller(x,autolag='AIC'); print(f"ADF  ({l}): stat={r[0]:.3f} p={r[1]:.4f}"); return r[1]
def kp(x,l):
    s,p,_,_=kpss(x,regression='c',nlags='auto'); print(f"KPSS ({l}): stat={s:.3f} p={p:.4f}"); return p
ts_d=np.diff(ts)
print("== Nível =="); adf(ts,'nível'); kp(ts,'nível')
print("\n== 1ª diferença =="); adf(ts_d,'1a dif'); kp(ts_d,'1a dif')
print("\n→ Série I(1): d=1")

fig,axes=plt.subplots(2,2,figsize=(11,6))
plot_acf(ts,lags=ML,ax=axes[0,0],color=AZUL,title='ACF — nível')
plot_pacf(ts,lags=ML,ax=axes[0,1],color=AZUL,title='PACF — nível',method='ywm')
plot_acf(ts_d,lags=ML,ax=axes[1,0],color=VERDE,title='ACF — 1ª diferença')
plot_pacf(ts_d,lags=ML,ax=axes[1,1],color=VERDE,title='PACF — 1ª diferença',method='ywm')
plt.tight_layout(); plt.savefig('f3_acfpacf.png', dpi=150, bbox_inches='tight'); plt.show()

## 4. SARIMA — seleção por AICc e diagnóstico

In [ ]:
specs=[(1,1,0,0,0,0),(0,1,1,0,0,0),(1,1,1,0,0,0),(2,1,0,0,0,0),(2,1,1,0,0,0),
       (1,1,0,1,0,0),(0,1,1,0,0,1),(1,1,1,1,0,0)]
res=[]
for p,d,q,P,D,Q in specs:
    try:
        mm=SARIMAX(ts,order=(p,d,q),seasonal_order=(P,D,Q,12),trend='n',
                   enforce_stationarity=False,enforce_invertibility=False).fit(disp=False)
        res.append({'Modelo':f'SARIMA({p},{d},{q})({P},{D},{Q})[12]','AIC':round(mm.aic,1),
                    'BIC':round(mm.bic,1),'AICc':round(mm.aicc,1),'_o':mm})
    except: pass
tab=pd.DataFrame(res).sort_values('AICc').reset_index(drop=True)
display(tab.drop(columns='_o'))
nome=tab.iloc[0]['Modelo']; fit=next(r['_o'] for r in res if r['Modelo']==nome)
print("Melhor:",nome)

fitted=np.array(fit.fittedvalues); resid=np.array(fit.resid); resid=resid[~np.isnan(resid)]
fig=plt.figure(figsize=(11,6.5)); gs=gridspec.GridSpec(2,3,figure=fig,hspace=0.4,wspace=0.32)
a1=fig.add_subplot(gs[0,:]); a2=fig.add_subplot(gs[1,0]); a3=fig.add_subplot(gs[1,1]); a4=fig.add_subplot(gs[1,2])
a1.plot(datas,ts,color=AZUL,lw=1.8,label='Observado'); a1.plot(datas,fitted,color=VERM,lw=1.4,ls='--',label='Ajustado')
a1.yaxis.set_major_formatter(fmt); a1.legend(); a1.set_title(f'{nome} — ajuste',fontweight='bold')
(o,r2),(sl,ic,rr)=stats.probplot(resid,dist='norm'); a2.scatter(o,r2,s=15,color=AZUL); a2.plot(o,sl*np.array(o)+ic,color=VERM); a2.set_title(f'QQ (r={rr:.3f})')
plot_acf(resid,lags=ML,ax=a3,color=VERDE,title='ACF resíduos')
a4.hist(resid,bins=9,color=AZUL,alpha=0.7,density=True,edgecolor='white'); a4.set_title('Histograma')
plt.savefig('f4_diag.png', dpi=150, bbox_inches='tight'); plt.show()

lb=acorr_ljungbox(resid,lags=[12],return_df=True); sw=stats.shapiro(resid); ap=het_arch(resid,nlags=4)[1]
mae=np.mean(np.abs(ts-fitted)); rmse=np.sqrt(np.mean((ts-fitted)**2)); mape=np.mean(np.abs((ts-fitted)/ts))*100
print(f"Ljung-Box(12) p={lb['lb_pvalue'].iloc[-1]:.3f} | Shapiro p={sw.pvalue:.4f} | ARCH p={ap:.3f}")
print(f"MAE=R${mae:.0f} RMSE=R${rmse:.0f} MAPE={mape:.1f}%")

## 4b. Exercício de previsão: 30 treino + 10 teste
Treina o SARIMA com as 30 primeiras observações e prevê as 10 seguintes, exibindo a tabela com observado, previsto e resíduo de previsão.

In [ ]:
NTR=30
trn3,te3 = ts[:NTR], ts[NTR:NTR+10]
datas_te3 = datas[NTR:NTR+10]
f3=SARIMAX(trn3,order=(1,1,0),seasonal_order=(1,0,0,12),trend='n',
           enforce_stationarity=False,enforce_invertibility=False).fit(disp=False)
fo=f3.get_forecast(steps=10); fc3=np.array(fo.predicted_mean); ci3=np.array(fo.conf_int(alpha=0.05))

tab=pd.DataFrame({
    'Mês':[d.strftime('%b/%Y') for d in datas_te3],
    'Observado':te3.round(0),
    'Previsto':fc3.round(0),
    'Resíduo':(te3-fc3).round(0),
    'Erro %':((te3-fc3)/te3*100).round(1),
})
display(tab)
mae3=np.mean(np.abs(te3-fc3)); rmse3=np.sqrt(np.mean((te3-fc3)**2)); mape3=np.mean(np.abs((te3-fc3)/te3))*100
dentro=int(np.sum((te3>=ci3[:,0])&(te3<=ci3[:,1])))
print(f"MAE=R${mae3:.0f}  RMSE=R${rmse3:.0f}  MAPE={mape3:.1f}%  | dentro do IC95%: {dentro}/10")

fig,ax=plt.subplots(figsize=(10,4))
ax.plot(datas[:NTR],trn3,color=AZUL,lw=1.7,label='Treino (30)')
ax.plot(datas_te3,te3,'o-',color='black',lw=1.8,ms=5,label='Observado (10)')
ax.plot(datas_te3,fc3,'s--',color=VERM,lw=1.6,ms=4,label=f'Previsto (MAE={mae3:.0f})')
ax.fill_between(datas_te3,ci3[:,0],ci3[:,1],alpha=0.15,color=VERM,label='IC 95%')
ax.axvline(datas.iloc[NTR-1],color='gray',ls=':',lw=1); ax.yaxis.set_major_formatter(fmt)
ax.legend(fontsize=8,ncol=2); ax.set_title('Previsão fora da amostra: 30 treino, 10 previsão',fontweight='bold')
plt.savefig('f7_prev3010.png', dpi=150, bbox_inches='tight'); plt.show()
print('Obs.: o treino termina no pico (mai/2025); o modelo superestima a queda posterior.')

## 5. Modelo híbrido SARIMA→BTVC (PyMC/MCMC)
A previsão do SARIMA entra como regressor com peso β(t) variável no tempo, aprendido via MCMC. O modelo bayesiano decide quanta confiança dar ao SARIMA em cada instante.

In [ ]:
# Híbrido SARIMA→BTVC — previsão out-of-sample (30 treino + 10), comparável à Seção 6
NTR=30; H=10
tr,te=ts[:NTR],ts[NTR:NTR+10]; datas_te=datas[NTR:NTR+10]
fit_s=SARIMAX(tr,order=(1,1,0),seasonal_order=(1,0,0,12),trend='n',
    enforce_stationarity=False,enforce_invertibility=False).fit(disp=False)
sar_in=np.array(fit_s.fittedvalues); sar_oos=np.array(fit_s.get_forecast(steps=H).predicted_mean)
m,s=tr.mean(),tr.std(); yn=(tr-m)/s; xn=(sar_in-m)/s
K=2; tt=np.arange(NTR,dtype=float)
F=np.column_stack([fn for k in range(1,K+1) for fn in [np.sin(2*np.pi*k*tt/12),np.cos(2*np.pi*k*tt/12)]])
with pm.Model() as hib:
    sl=pm.HalfNormal('sl',0.2); sb=pm.HalfNormal('sb',0.1); sg=pm.HalfNormal('sg',0.1)
    so=pm.HalfNormal('so',0.3); nu=pm.Gamma('nu',2,0.2)
    l0=pm.Normal('l0',float(yn[0]),1); li=pm.Normal('li',0,sl,shape=NTR-1)
    lev=pm.Deterministic('lev',pt.concatenate([[l0],l0+pt.cumsum(li)]))
    b0=pm.Normal('b0',1,0.5); bi=pm.Normal('bi',0,sb,shape=NTR-1)
    beta=pm.Deterministic('beta',pt.concatenate([[b0],b0+pt.cumsum(bi)]))
    g0=pm.Normal('g0',0,0.5,shape=2*K); gi=pm.Normal('gi',0,sg,shape=(NTR-1,2*K))
    gw=pm.Deterministic('gw',pt.concatenate([g0[None,:],g0[None,:]+pt.cumsum(gi,axis=0)],axis=0))
    mu=lev+beta*xn+pt.sum(gw*pt.as_tensor_variable(F),axis=1)
    pm.StudentT('obs',nu=nu,mu=mu,sigma=so,observed=yn)
    idata=pm.sample(800,tune=800,chains=2,cores=1,target_accept=0.92,progressbar=True,random_seed=SEED)
ll=idata.posterior['lev'].values.reshape(-1,NTR)[:,-1]
bl=idata.posterior['beta'].values.reshape(-1,NTR)[:,-1]
gl=idata.posterior['gw'].values.reshape(-1,NTR,2*K)[:,-1,:]
tf=np.arange(NTR,NTR+H,dtype=float)
Ff=np.column_stack([fn for k in range(1,K+1) for fn in [np.sin(2*np.pi*k*tf/12),np.cos(2*np.pi*k*tf/12)]])
ps=ll[:,None]+bl[:,None]*((sar_oos-m)/s)[None,:]+(gl@Ff.T)
pred=ps.mean(0)*s+m; plo=np.percentile(ps,2.5,0)*s+m; phi=np.percentile(ps,97.5,0)*s+m
tab_h=pd.DataFrame({'Mês':[d.strftime('%b/%Y') for d in datas_te],
    'Observado':te.round(0),'Previsto':pred.round(0),
    'Resíduo':(te-pred).round(0),'Erro %':((te-pred)/te*100).round(1)})
display(tab_h)
mae_h=np.mean(np.abs(te-pred)); mape_h=np.mean(np.abs((te-pred)/te))*100
print(f"Híbrido OOS: MAE=R${mae_h:.0f}  MAPE={mape_h:.1f}%  (SARIMA no mesmo teste: MAE=R$269, MAPE=29.9%)")
fig,ax=plt.subplots(figsize=(10,4))
ax.plot(datas[:NTR],tr,color=AZUL,lw=1.7,label='Treino (30)')
ax.plot(datas_te,te,'o-',color='black',lw=1.8,ms=5,label='Observado (10)')
ax.plot(datas_te,pred,'^--',color=VERDE,lw=1.6,ms=4,label=f'Híbrido (MAE={mae_h:.0f})')
ax.fill_between(datas_te,plo,phi,alpha=0.15,color=VERDE,label='IC 95%')
ax.axvline(datas.iloc[NTR-1],color='gray',ls=':',lw=1); ax.yaxis.set_major_formatter(fmt)
ax.legend(fontsize=8,ncol=2); ax.set_title('Previsão fora da amostra do híbrido (30 treino, 10 previsão)',fontweight='bold')
plt.savefig('f8_hibrido.png', dpi=150, bbox_inches='tight'); plt.show()

## 6. Modelo hierárquico de safra (PyMC/MCMC)
Sem fazer a média: cada observação (mês × safra) é mantida, e a safra entra como efeito aleatório. Quantifica a variabilidade ENTRE safras que a média esconde.

In [ ]:
# Modelo hierárquico de safra (efeitos aleatórios)
men=raw.copy()
meses=sorted(men['am'].unique()); mi={x:i for i,x in enumerate(meses)}
safras=sorted(men['safra'].dropna().unique()); si={x:i for i,x in enumerate(safras)}
men['t']=men['am'].map(mi); men['s']=men['safra'].map(si); men['data_m']=men['am'].dt.to_timestamp()
yv=men['preco_rs_ton'].values; tv=men['t'].values; sv=men['s'].values
nm=len(meses); nsf=len(safras); mh,sh=yv.mean(),yv.std(); ynh=(yv-mh)/sh
with pm.Model() as hier:
    slv=pm.HalfNormal('slv',0.3); l0=pm.Normal('l0',float(ynh[0]),1); li=pm.Normal('li',0,slv,shape=nm-1)
    niv=pm.Deterministic('niv',pt.concatenate([[l0],l0+pt.cumsum(li)]))
    ssf=pm.HalfNormal('ssf',0.5); al=pm.Normal('al',0,ssf,shape=nsf); sob=pm.HalfNormal('sob',0.3)
    pm.StudentT('obs',nu=5,mu=niv[tv]+al[sv],sigma=sob,observed=ynh)
    idh=pm.sample(700,tune=700,chains=2,cores=1,target_accept=0.9,progressbar=True,random_seed=SEED)
niv_mn=idh.posterior['niv'].values.reshape(-1,nm).mean(0)*sh+mh
ef=idh.posterior['al'].values.reshape(-1,nsf).mean(0)*sh; sigsf=idh.posterior['ssf'].values.mean()*sh
datas_m=[m.to_timestamp() for m in meses]
print("Efeito de cada safra (desvio do nível comum):")
for x,e in zip(safras,ef): print(f"  Safra {x}: {e:+.0f} R$/ton")
print(f"Variabilidade ENTRE safras (sigma_safra) = R${sigsf:.0f}/ton")

fig,axes=plt.subplots(1,2,figsize=(12,4.2),gridspec_kw={'width_ratios':[1.6,1]})
cores_s={s:c for s,c in zip(safras,[AZUL,VERDE,AMAR])}
for s in safras:
    sub=men[men['safra']==s]
    axes[0].scatter(sub['data_m'],sub['preco_rs_ton'],s=30,color=cores_s[s],label=f'Safra {s}',alpha=0.8,edgecolor='white',zorder=3)
axes[0].plot(datas_m,niv_mn,color=VERM,lw=2.2,label=r'Nível comum $\mu_t$',zorder=2)
axes[0].yaxis.set_major_formatter(fmt); axes[0].legend(fontsize=8)
axes[0].set_title('(a) Observações por safra e nível comum',fontweight='bold',fontsize=10); axes[0].set_ylabel('R$/ton')
axes[1].barh([f'Safra {s}' for s in safras],ef,color=[VERDE if e>=0 else VERM for e in ef],alpha=0.8,edgecolor='white')
axes[1].axvline(0,color='black',lw=1)
for i2,e in enumerate(ef): axes[1].text(e+(14 if e>=0 else -14),i2,f'{e:+.0f}',va='center',ha='left' if e>=0 else 'right',fontsize=9,fontweight='bold')
axes[1].margins(x=0.18)
axes[1].set_title('(b) Efeito por safra\n($\\sigma_{safra}$ = R\$'+f'{sigsf:.0f}/ton)',fontweight='bold',fontsize=10)
axes[1].set_xlabel('Desvio do nível comum (R$/ton)')
plt.tight_layout(); plt.savefig('f6_hier.png', dpi=150, bbox_inches='tight'); plt.show()

## 7. Validação out-of-sample (comparação justa)

In [ ]:
H=6; trn,te=ts[:-H],ts[-H:]
ft=SARIMAX(trn,order=(1,1,0),seasonal_order=(1,0,0,12),trend='n',
           enforce_stationarity=False,enforce_invertibility=False).fit(disp=False)
fc=np.array(ft.get_forecast(steps=H).predicted_mean); mae_oos=np.mean(np.abs(te-fc))

# BTVC OOS
tt=np.arange(len(trn),dtype=float)
Ft=np.column_stack([fn for k in range(1,K+1) for fn in [np.sin(2*np.pi*k*tt/12),np.cos(2*np.pi*k*tt/12)]])
m2,s2=trn.mean(),trn.std(); yn2=(trn-m2)/s2
with pm.Model():
    sl=pm.HalfNormal('sl',0.3); sz=pm.HalfNormal('sz',0.15); so=pm.HalfNormal('so',0.3); nu=pm.Gamma('nu',2,0.2)
    l0=pm.Normal('l0',float(yn2[0]),1); li=pm.Normal('li',0,sl,shape=len(trn)-1)
    lev=pm.Deterministic('lev',pt.concatenate([[l0],l0+pt.cumsum(li)]))
    b0=pm.Normal('b0',0,0.5,shape=2*K); bi=pm.Normal('bi',0,sz,shape=(len(trn)-1,2*K))
    bw=pm.Deterministic('bw',pt.concatenate([b0[None,:],b0[None,:]+pt.cumsum(bi,axis=0)],axis=0))
    pm.StudentT('obs',nu=nu,mu=lev+pt.sum(bw*pt.as_tensor_variable(Ft),axis=1),sigma=so,observed=yn2)
    ido=pm.sample(500,tune=500,chains=2,cores=1,target_accept=0.9,progressbar=False,random_seed=SEED)
ll=ido.posterior['lev'].values.reshape(-1,len(trn))[:,-1]
bl=ido.posterior['bw'].values.reshape(-1,len(trn),2*K)[:,-1,:]
tf=np.arange(len(trn),len(trn)+H,dtype=float)
Ff=np.column_stack([fn for k in range(1,K+1) for fn in [np.sin(2*np.pi*k*tf/12),np.cos(2*np.pi*k*tf/12)]])
fb=(ll[:,None]+bl@Ff.T).mean(0)*s2+m2; mae_b_oos=np.mean(np.abs(te-fb))

fig,ax=plt.subplots(figsize=(10,4))
ax.plot(datas[:-H],trn,color=AZUL,lw=1.6,label='Treino')
ax.plot(datas[-H:],te,'o-',color='black',lw=1.8,ms=5,label='Teste real')
ax.plot(datas[-H:],fc,'s--',color=VERM,lw=1.6,ms=4,label=f'SARIMA (MAE={mae_oos:.0f})')
ax.plot(datas[-H:],fb,'^--',color=VERDE,lw=1.6,ms=4,label=f'BTVC (MAE={mae_b_oos:.0f})')
ax.axvline(datas.iloc[-H],color='gray',ls=':',lw=1); ax.yaxis.set_major_formatter(fmt); ax.legend()
ax.set_title('Validação out-of-sample (últimos 6 meses)',fontweight='bold'); plt.show()
print(f"SARIMA OOS MAE=R${mae_oos:.0f} | BTVC OOS MAE=R${mae_b_oos:.0f}")
print('→ SARIMA vence na previsão de rotina; BTVC só compensa sob choque estrutural.')

## 8. Conclusão
- **SARIMA(1,1,0)(1,0,0)[12]**: modelo principal, resíduos ruído branco, sem ARCH.
- **BTVC**: sobreajusta dentro da amostra; pior fora dela; útil só sob choque.
- **Hierárquico**: σ_safra ≈ R$207/ton justifica criticamente a média das safras.

Recomendação: SARIMA para rotina; BTVC como contingência para instabilidade.

## Exportar figuras
Salva e baixa todas as figuras em PNG para usar no relatório LaTeX.

In [ ]:
# === Figuras salvas na pasta atual (para subir no Overleaf com o .tex) ===
import os
figs = ['f1_serie.png','f2_stl.png','f3_acfpacf.png','f4_diag.png',
        'f7_prev3010.png','f8_hibrido.png','f6_hier.png']
existentes = [f for f in figs if os.path.exists(f)]
print("Figuras salvas na pasta atual:")
for f in existentes:
    print("  -", f)
print("\nNo Colab, abra o painel de Arquivos (ícone de pasta à esquerda) para baixá-las.")